In [ ]:

# =============================================================================
# XGBOOST V2 FORECASTING
# CELL 1 — ENVIRONMENT + FROZEN DATASET VERIFICATION
#
# READ-ONLY: NO MODEL TRAINING
# =============================================================================

from pathlib import Path
import hashlib
import json
import sys

import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import average_precision_score

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/network-attack-forecasting"
)

DATA_DIR = (
    PROJECT_ROOT
    / "Synthetic_NAF_v2"
    / "release_candidate_corrected"
)

FREEZE_DIR = (
    PROJECT_ROOT
    / "Synthetic_NAF_v2"
    / "FINAL_RELEASE_FREEZE"
)

TRAIN_FILE = DATA_DIR / "windows_W10_TRAIN.npz"

METADATA_FILE = (
    DATA_DIR
    / "episode_metadata_DO_NOT_USE_AS_PREDICTORS.csv"
)

CONTRACT_FILE = (
    DATA_DIR
    / "FEATURE_AND_DATASET_CONTRACT.json"
)

RELEASE_FILE = (
    FREEZE_DIR
    / "Synthetic_NAF_v2_RELEASE_MANIFEST.json"
)

LOCK_FILE = (
    FREEZE_DIR
    / "Synthetic_NAF_v2_FREEZE_LOCK.json"
)

HASH_FILE = DATA_DIR / "SHA256SUMS.json"

if not Path("/content/drive/MyDrive").exists():
    from google.colab import drive
    drive.mount("/content/drive")

required_files = [
    TRAIN_FILE,
    METADATA_FILE,
    CONTRACT_FILE,
    RELEASE_FILE,
    LOCK_FILE,
    HASH_FILE
]

for path in required_files:
    if not path.is_file():
        raise FileNotFoundError(
            f"Required frozen dataset file is missing: {path}"
        )

def sha256_file(path):
    digest = hashlib.sha256()

    with open(path, "rb") as f:
        for block in iter(
            lambda: f.read(1024 * 1024),
            b""
        ):
            digest.update(block)

    return digest.hexdigest()


def canonical_hash(obj):
    payload = json.dumps(
        obj,
        sort_keys=True,
        separators=(",", ":"),
        ensure_ascii=False
    ).encode("utf-8")

    return hashlib.sha256(payload).hexdigest()


with open(CONTRACT_FILE) as f:
    CONTRACT = json.load(f)

with open(RELEASE_FILE) as f:
    RELEASE = json.load(f)

with open(LOCK_FILE) as f:
    LOCK = json.load(f)

with open(HASH_FILE) as f:
    RECORDED_HASHES = json.load(f)

assert RELEASE["release_version"] == "2.0.0"
assert RELEASE["release_status"] == "FROZEN_CONTROLLED_SYNTHETIC_BENCHMARK"

assert LOCK["manifest_sha256_canonical"] == canonical_hash(RELEASE)

assert LOCK["version"] == "2.0.0"

# Verify all frozen dataset artifact hashes.
for filename, expected_hash in RECORDED_HASHES.items():
    actual_hash = sha256_file(DATA_DIR / filename)

    if actual_hash != expected_hash:
        raise RuntimeError(
            f"Frozen dataset integrity failure: {filename}"
        )

# Verify the manifest includes correct dataset artifact hashes.
for filename, expected_hash in RELEASE["data_integrity"]["file_sha256"].items():
    actual_hash = sha256_file(DATA_DIR / filename)

    if actual_hash != expected_hash:
        raise RuntimeError(
            f"Release manifest hash mismatch: {filename}"
        )

FEATURES = list(CONTRACT["features"])
FAMILIES = list(CONTRACT["attack_families"])

HORIZONS = [5, 10, 15]
HORIZON_TO_COLUMN = {5: 0, 10: 1, 15: 2}

assert FEATURES == RELEASE["feature_contract"]["predictors"]
assert FAMILIES == RELEASE["feature_contract"]["attack_families"]
assert CONTRACT["history_minutes"] == 10
assert CONTRACT["forecast_horizons"] == HORIZONS

print("=" * 90)
print("XGBOOST V2 — FROZEN DATASET VERIFIED")
print("=" * 90)

print("Release:", RELEASE["release_name"])
print("Version:", RELEASE["release_version"])
print("Dataset SHA256 checks: PASS")
print("Features:", len(FEATURES))
print("Attack families:", FAMILIES)
print("Horizons:", HORIZONS)
print("TRAIN file:", TRAIN_FILE)
print("Model training: NOT STARTED")
print("Official VALIDATION / TEST: NOT LOADED")


In [ ]:

# =============================================================================
# CELL 2 — LOAD TRAIN DATA AND VERIFY WINDOW / TARGET ALIGNMENT
#
# READ-ONLY: NO MODEL TRAINING
# =============================================================================

with np.load(TRAIN_FILE, allow_pickle=False) as pack:
    X_SEQ = pack["X"].astype(np.float32)
    Y_HORIZONS = pack["y_horizons"].astype(np.int8)

    ONSET_CLASS = pack["onset_class"].astype(np.int16)
    FAMILY_CLASS = pack["family_class"].astype(np.int8)

    EPISODE_ID = pd.Series(
        pack["episode_id"]
    ).astype(str).to_numpy()

    MINUTE = pack["minute"].astype(np.int16)

meta = pd.read_csv(METADATA_FILE)
meta["Episode_ID"] = meta["Episode_ID"].astype(str)

train_meta = (
    meta.loc[meta["Split"] == "TRAIN"]
    .set_index("Episode_ID")
)

assert train_meta.index.is_unique
assert len(X_SEQ) == len(EPISODE_ID)

assert X_SEQ.shape[1:] == (10, 14)
assert Y_HORIZONS.shape == (len(X_SEQ), 3)

assert set(EPISODE_ID).issubset(set(train_meta.index))

assert np.isfinite(X_SEQ).all()
assert (X_SEQ >= 0).all()

assert np.array_equal(
    Y_HORIZONS[:, 0],
    ((ONSET_CLASS >= 1) & (ONSET_CLASS <= 5)).astype(np.int8)
)

assert np.array_equal(
    Y_HORIZONS[:, 1],
    ((ONSET_CLASS >= 1) & (ONSET_CLASS <= 10)).astype(np.int8)
)

assert np.array_equal(
    Y_HORIZONS[:, 2],
    ((ONSET_CLASS >= 1) & (ONSET_CLASS <= 15)).astype(np.int8)
)

SOURCE = train_meta.loc[EPISODE_ID, "Source"].astype(str).to_numpy()

CAL_GROUP = (
    train_meta.loc[
        EPISODE_ID,
        "Calibration_Group_ID"
    ]
    .astype(str)
    .to_numpy()
)

CV_GROUPS = np.array([
    f"{source}::{group}"
    for source, group in zip(SOURCE, CAL_GROUP)
])

assert len(np.unique(EPISODE_ID)) == 1400
assert len(np.unique(CV_GROUPS)) == 35

assert not pd.MultiIndex.from_arrays([
    EPISODE_ID,
    MINUTE
]).duplicated().any()

print("=" * 90)
print("CELL 2 — TRAIN DATA VERIFIED")
print("=" * 90)

print("X sequence shape:", X_SEQ.shape)
print("Forecasting targets:", Y_HORIZONS.shape)
print("Unique episodes:", len(np.unique(EPISODE_ID)))
print("Calibration CV groups:", len(np.unique(CV_GROUPS)))

print("\nHorizon prevalence:")
for h, col in HORIZON_TO_COLUMN.items():
    print(
        f"H{h}: {Y_HORIZONS[:, col].mean():.6f}"
    )

print("\nTraining status: NOT STARTED")


In [ ]:

# =============================================================================
# CELL 3 — BUILD FLAT140 TEMPORAL FEATURES
#
# Input:  (N, 10, 14)
# Output: (N, 140)
#
# Oldest minute first, newest minute last.
# No future information or metadata.
#
# READ-ONLY: NO MODEL TRAINING
# =============================================================================

N, WINDOW_LENGTH, N_FEATURES = X_SEQ.shape

assert WINDOW_LENGTH == 10
assert N_FEATURES == 14

X_FLAT = np.log1p(
    X_SEQ
).reshape(N, WINDOW_LENGTH * N_FEATURES)

X_FLAT = np.ascontiguousarray(
    X_FLAT,
    dtype=np.float32
)

TEMPORAL_FEATURE_NAMES = [
    f"t_minus_{9 - step}__{feature}"
    for step in range(10)
    for feature in FEATURES
]

assert X_FLAT.shape == (len(X_SEQ), 140)
assert len(TEMPORAL_FEATURE_NAMES) == 140
assert len(set(TEMPORAL_FEATURE_NAMES)) == 140
assert np.isfinite(X_FLAT).all()

# Independently verify the flattening order.
sample_indices = np.unique(
    np.linspace(
        0,
        len(X_SEQ) - 1,
        num=min(100, len(X_SEQ)),
        dtype=int
    )
)

for idx in sample_indices:
    expected = np.log1p(X_SEQ[idx]).reshape(140)

    if not np.allclose(
        X_FLAT[idx],
        expected,
        rtol=1e-6,
        atol=1e-6
    ):
        raise RuntimeError(
            f"FLAT140 temporal reconstruction failed: {idx}"
        )

# These are separate forecasting targets,
# not a combined multiclass classification target.
Y_H5 = Y_HORIZONS[:, 0].astype(np.int8)
Y_H10 = Y_HORIZONS[:, 1].astype(np.int8)
Y_H15 = Y_HORIZONS[:, 2].astype(np.int8)

print("=" * 90)
print("CELL 3 — FLAT140 FEATURES READY")
print("=" * 90)

print("Original input:", X_SEQ.shape)
print("XGBoost input:", X_FLAT.shape)
print("Feature count:", len(TEMPORAL_FEATURE_NAMES))

print("\nExample temporal features:")
print(TEMPORAL_FEATURE_NAMES[:7])
print("...")
print(TEMPORAL_FEATURE_NAMES[-7:])

print("\nH5 positive windows:", int(Y_H5.sum()))
print("H10 positive windows:", int(Y_H10.sum()))
print("H15 positive windows:", int(Y_H15.sum()))

print("\nNo model training performed.")


In [ ]:

# =============================================================================
# CELL 4 — TRAIN-ONLY GROUP-DISJOINT CROSS-VALIDATION
#
# No shared calibration group between fold TRAIN and fold HOLDOUT.
# No official VALIDATION or TEST access.
#
# READ-ONLY: NO MODEL TRAINING
# =============================================================================

from sklearn.model_selection import GroupKFold

N_SPLITS = 5

unique_groups = np.unique(CV_GROUPS)

if len(unique_groups) < N_SPLITS:
    raise RuntimeError(
        "Insufficient calibration groups for requested folds."
    )

FOLD_ASSIGNMENTS = np.full(
    len(X_FLAT),
    -1,
    dtype=np.int8
)

CV_FOLDS = []

splitter = GroupKFold(n_splits=N_SPLITS)

FAMILY_BY_WINDOW = (
    train_meta.loc[EPISODE_ID, "Family"]
    .astype(str)
    .to_numpy()
)

summary_rows = []

for fold_id, (fit_idx, hold_idx) in enumerate(
    splitter.split(
        X_FLAT,
        Y_H10,
        groups=CV_GROUPS
    ),
    start=1
):

    fit_groups = set(CV_GROUPS[fit_idx])
    hold_groups = set(CV_GROUPS[hold_idx])

    fit_episodes = set(EPISODE_ID[fit_idx])
    hold_episodes = set(EPISODE_ID[hold_idx])

    assert fit_groups.isdisjoint(hold_groups)
    assert fit_episodes.isdisjoint(hold_episodes)

    FOLD_ASSIGNMENTS[hold_idx] = fold_id

    CV_FOLDS.append((fit_idx, hold_idx))

    row = {
        "Fold": fold_id,
        "Fit_Windows": len(fit_idx),
        "Holdout_Windows": len(hold_idx),
        "Fit_Groups": len(fit_groups),
        "Holdout_Groups": len(hold_groups),
        "Fit_Episodes": len(fit_episodes),
        "Holdout_Episodes": len(hold_episodes),
    }

    for horizon, column in HORIZON_TO_COLUMN.items():

        y_fit = Y_HORIZONS[fit_idx, column]
        y_hold = Y_HORIZONS[hold_idx, column]

        row[f"H{horizon}_Fit_Positive"] = int(y_fit.sum())
        row[f"H{horizon}_Holdout_Positive"] = int(y_hold.sum())
        row[f"H{horizon}_Holdout_Prevalence"] = float(
            y_hold.mean()
        )

        if len(np.unique(y_fit)) != 2:
            raise RuntimeError(
                f"Fold {fold_id} H{horizon}: "
                "fit partition has only one class."
            )

        if len(np.unique(y_hold)) != 2:
            raise RuntimeError(
                f"Fold {fold_id} H{horizon}: "
                "holdout partition has only one class."
            )

    for source in sorted(np.unique(SOURCE)):

        row[f"Holdout_Source_{source}"] = int(
            (SOURCE[hold_idx] == source).sum()
        )

    for family in FAMILIES:

        count = len(
            set(
                EPISODE_ID[hold_idx][
                    FAMILY_BY_WINDOW[hold_idx] == family
                ]
            )
        )

        row[f"Holdout_Episodes_{family}"] = count

    summary_rows.append(row)

assert (FOLD_ASSIGNMENTS >= 1).all()

# All windows from an episode must belong to the same fold.
episode_fold_counts = (
    pd.DataFrame({
        "episode": EPISODE_ID,
        "fold": FOLD_ASSIGNMENTS
    })
    .groupby("episode")["fold"]
    .nunique()
)

assert (episode_fold_counts == 1).all()

group_fold_counts = (
    pd.DataFrame({
        "group": CV_GROUPS,
        "fold": FOLD_ASSIGNMENTS
    })
    .groupby("group")["fold"]
    .nunique()
)

assert (group_fold_counts == 1).all()

CV_SUMMARY = pd.DataFrame(summary_rows)

print("=" * 90)
print("CELL 4 — GROUP-DISJOINT CV READY")
print("=" * 90)

print(
    CV_SUMMARY[
        [
            "Fold",
            "Fit_Windows",
            "Holdout_Windows",
            "Fit_Groups",
            "Holdout_Groups",
            "H5_Holdout_Positive",
            "H10_Holdout_Positive",
            "H15_Holdout_Positive",
        ]
    ].to_string(index=False)
)

print("\nAttack-family representation:")
print(
    CV_SUMMARY[
        ["Fold"] + [
            f"Holdout_Episodes_{family}"
            for family in FAMILIES
        ]
    ].to_string(index=False)
)

print("\nCalibration-group overlap: ZERO")
print("Episode overlap: ZERO")
print("Official VALIDATION / TEST: NOT USED")
print("Model training: NOT STARTED")


In [ ]:

# =============================================================================
# CELL 5 — OPTUNA XGBOOST FLAT140 OPTIMIZATION
#
# THIS CELL TRAINS XGBOOST MODELS WHEN EXECUTED.
#
# TRAIN-only:
#   H5  — 25 trials, 5 group-disjoint folds
#   H10 — 25 trials, 5 group-disjoint folds
#
# No official VALIDATION or TEST access.
# No final full-TRAIN model fitting.
# No changes to the frozen dataset.
# =============================================================================

import sys
import subprocess
import importlib.util

if importlib.util.find_spec("optuna") is None:
    subprocess.check_call([
        sys.executable, "-m", "pip",
        "-q", "install", "optuna"
    ])

if importlib.util.find_spec("xgboost") is None:
    subprocess.check_call([
        sys.executable, "-m", "pip",
        "-q", "install", "xgboost"
    ])

import optuna
import xgboost as xgb
import numpy as np
import pandas as pd
import json
import hashlib
import gc

from datetime import datetime, timezone
from pathlib import Path
from sklearn.metrics import average_precision_score

# =============================================================================
# 5A — ENSURE PREPARATION CELLS HAVE RUN
# =============================================================================

required_variables = [
    "X_FLAT",
    "Y_HORIZONS",
    "CV_FOLDS",
    "CV_GROUPS",
    "EPISODE_ID",
    "SOURCE",
    "HORIZON_TO_COLUMN",
    "DATA_DIR",
    "FREEZE_DIR",
    "FEATURES",
    "TEMPORAL_FEATURE_NAMES"
]

missing = [
    name for name in required_variables
    if name not in globals()
]

if missing:
    raise RuntimeError(
        f"Run Cells 1–4 before Cell 5. Missing: {missing}"
    )

assert X_FLAT.shape == (88186, 140)
assert Y_HORIZONS.shape == (88186, 3)
assert len(CV_FOLDS) == 5

assert HORIZON_TO_COLUMN == {
    5: 0,
    10: 1,
    15: 2
}

assert np.isfinite(X_FLAT).all()

# =============================================================================
# 5B — FREEZE OPTIMIZATION SETTINGS
# =============================================================================

MASTER_SEED = 20261022

OPTIMIZATION_HORIZONS = [5, 10]

N_TRIALS = 25
N_SPLITS = 5

MAX_BOOST_ROUNDS = 1800
EARLY_STOPPING_ROUNDS = 100

# CPU is the reproducible default.
# Set this to "cuda" only if Colab provides a compatible GPU
# and the installed XGBoost supports GPU training.
DEVICE = "cpu"

OPT_ROOT = (
    PROJECT_ROOT
    / "Synthetic_NAF_v2"
    / "models"
    / "XGBoost_V2_FLAT140"
    / "train_cv_optimization"
)

OPT_ROOT.mkdir(parents=True, exist_ok=True)

# Optuna studies are persistent.
# This prevents losing completed trials after runtime interruption.

SEARCH_SPACE_VERSION = "XGB_V2_FLAT140_SEARCH_V1"

OPTUNA_STARTUP_TRIALS = 8

print("=" * 105)
print("XGBOOST V2 — TRAIN-ONLY OPTUNA OPTIMIZATION")
print("=" * 105)

print("XGBoost version:", xgb.__version__)
print("Optuna version:", optuna.__version__)
print("Device:", DEVICE)
print("Horizons:", OPTIMIZATION_HORIZONS)
print("Trials per horizon:", N_TRIALS)
print("Folds per trial:", N_SPLITS)
print("Maximum boosting rounds:", MAX_BOOST_ROUNDS)
print("Early stopping patience:", EARLY_STOPPING_ROUNDS)
print("Output:", OPT_ROOT)

# =============================================================================
# 5C — CHECK SOURCE COVERAGE IN EVERY FOLD
# =============================================================================

all_sources = set(np.unique(SOURCE))

for fold_number, (fit_idx, hold_idx) in enumerate(
    CV_FOLDS, start=1
):

    fit_sources = set(SOURCE[fit_idx])
    hold_sources = set(SOURCE[hold_idx])

    if fit_sources != all_sources:
        raise RuntimeError(
            f"Fold {fold_number}: fitting source coverage incomplete. "
            f"Actual: {fit_sources}"
        )

    if hold_sources != all_sources:
        raise RuntimeError(
            f"Fold {fold_number}: holdout source coverage incomplete. "
            f"Actual: {hold_sources}"
        )

    fit_groups = set(CV_GROUPS[fit_idx])
    hold_groups = set(CV_GROUPS[hold_idx])

    assert fit_groups.isdisjoint(hold_groups)

    assert set(
        EPISODE_ID[fit_idx]
    ).isdisjoint(
        set(EPISODE_ID[hold_idx])
    )

print("\n✓ Every fold contains both sources")
print("✓ No shared calibration groups")
print("✓ No shared episode IDs")

# =============================================================================
# 5D — FIXED FEATURE / FOLD FINGERPRINT
# =============================================================================

def hash_ndarray(array):
    a = np.ascontiguousarray(array)
    h = hashlib.sha256()
    h.update(str(a.shape).encode())
    h.update(str(a.dtype).encode())
    h.update(a.tobytes())
    return h.hexdigest()


fold_payload = [
    {
        "fold": i,
        "fit_groups": sorted(
            set(CV_GROUPS[fit_idx])
        ),
        "hold_groups": sorted(
            set(CV_GROUPS[hold_idx])
        )
    }
    for i, (fit_idx, hold_idx) in enumerate(CV_FOLDS, 1)
]

fold_fingerprint = hashlib.sha256(
    json.dumps(
        fold_payload, sort_keys=True
    ).encode()
).hexdigest()

data_fingerprint = hash_ndarray(X_FLAT)

config = {
    "dataset": "Synthetic_NAF_v2",
    "dataset_version": "2.0.0",
    "representation": "FLAT140_LOG1P",
    "optimization_horizons": OPTIMIZATION_HORIZONS,
    "trials_per_horizon": N_TRIALS,
    "folds": N_SPLITS,
    "max_boost_rounds": MAX_BOOST_ROUNDS,
    "early_stopping_rounds": EARLY_STOPPING_ROUNDS,
    "device": DEVICE,
    "master_seed": MASTER_SEED,
    "search_space_version": SEARCH_SPACE_VERSION,
    "feature_names": TEMPORAL_FEATURE_NAMES,
    "fold_fingerprint": fold_fingerprint,
    "data_fingerprint": data_fingerprint,
    "xgboost_version": xgb.__version__,
    "optuna_version": optuna.__version__
}

config_path = OPT_ROOT / "OPTIMIZATION_PROTOCOL.json"

if config_path.exists():
    existing = json.loads(config_path.read_text())

    if existing != config:
        raise RuntimeError(
            "Existing Optuna optimization protocol differs. "
            "Do not mix trials from different configurations."
        )

else:
    config_path.write_text(
        json.dumps(config, indent=2)
    )

print("✓ Optimization protocol recorded")

# =============================================================================
# 5E — PRECONSTRUCT TRAIN FOLD MATRICES
# =============================================================================

FOLD_MATRICES = {}

for horizon in OPTIMIZATION_HORIZONS:

    target_col = HORIZON_TO_COLUMN[horizon]

    target = Y_HORIZONS[:, target_col].astype(
        np.int8
    )

    fold_data = []

    for fold_number, (fit_idx, hold_idx) in enumerate(
        CV_FOLDS, start=1
    ):

        y_fit = target[fit_idx]
        y_hold = target[hold_idx]

        if (
            len(np.unique(y_fit)) != 2
            or len(np.unique(y_hold)) != 2
        ):
            raise RuntimeError(
                f"H{horizon} fold {fold_number} contains one class."
            )

        dfit = xgb.DMatrix(
            X_FLAT[fit_idx],
            label=y_fit,
            feature_names=TEMPORAL_FEATURE_NAMES
        )

        dhold = xgb.DMatrix(
            X_FLAT[hold_idx],
            label=y_hold,
            feature_names=TEMPORAL_FEATURE_NAMES
        )

        fold_data.append({
            "fold": fold_number,
            "dfit": dfit,
            "dhold": dhold,
            "y_hold": y_hold,
            "positive_prevalence": float(
                y_hold.mean()
            )
        })

    FOLD_MATRICES[horizon] = fold_data

print("✓ TRAIN-only fold matrices prepared")

# =============================================================================
# 5F — DEFINE OPTUNA SEARCH SPACE
# =============================================================================

def suggest_params(trial, horizon):

    return {
        "objective": "binary:logistic",
        "eval_metric": "aucpr",

        "tree_method": "hist",
        "device": DEVICE,

        "max_depth": trial.suggest_int(
            "max_depth", 2, 7
        ),

        "eta": trial.suggest_float(
            "eta", 0.015, 0.20, log=True
        ),

        "min_child_weight": trial.suggest_float(
            "min_child_weight", 1.0, 20.0, log=True
        ),

        "subsample": trial.suggest_float(
            "subsample", 0.65, 1.0
        ),

        "colsample_bytree": trial.suggest_float(
            "colsample_bytree", 0.65, 1.0
        ),

        "gamma": trial.suggest_float(
            "gamma", 0.0, 5.0
        ),

        "lambda": trial.suggest_float(
            "lambda", 1e-2, 30.0, log=True
        ),

        "alpha": trial.suggest_float(
            "alpha", 1e-3, 10.0, log=True
        ),

        "max_bin": 256,
        "nthread": 2,
        "seed": MASTER_SEED + horizon,
        "verbosity": 0
    }


# =============================================================================
# 5G — OPTUNA OBJECTIVE
# =============================================================================

def make_objective(horizon):

    fold_data = FOLD_MATRICES[horizon]

    def objective(trial):

        params = suggest_params(
            trial, horizon
        )

        fold_aps = []
        fold_rounds = []
        fold_results = []

        for item in fold_data:

            fold_number = item["fold"]

            model = xgb.train(
                params=params,
                dtrain=item["dfit"],
                num_boost_round=MAX_BOOST_ROUNDS,
                evals=[
                    (item["dhold"], "fold_holdout")
                ],
                early_stopping_rounds=EARLY_STOPPING_ROUNDS,
                verbose_eval=False
            )

            best_round = int(
                model.best_iteration + 1
            )

            # Native xgboost.train can retain post-best
            # iterations. Predict only through best_iteration.
            predictions = model.predict(
                item["dhold"],
                iteration_range=(0, best_round)
            )

            ap = float(
                average_precision_score(
                    item["y_hold"],
                    predictions
                )
            )

            fold_aps.append(ap)
            fold_rounds.append(best_round)

            fold_results.append({
                "fold": fold_number,
                "average_precision": ap,
                "best_rounds": best_round,
                "prevalence": item["positive_prevalence"]
            })

            del model, predictions
            gc.collect()

        mean_ap = float(
            np.mean(fold_aps)
        )

        std_ap = float(
            np.std(fold_aps, ddof=1)
        )

        median_rounds = int(
            np.median(fold_rounds)
        )

        trial.set_user_attr(
            "fold_results", fold_results
        )

        trial.set_user_attr(
            "fold_ap_std", std_ap
        )

        trial.set_user_attr(
            "fold_best_rounds", fold_rounds
        )

        trial.set_user_attr(
            "median_best_rounds", median_rounds
        )

        print(
            f"H{horizon} | "
            f"Trial {trial.number:02d} | "
            f"Mean AP={mean_ap:.5f} | "
            f"SD={std_ap:.5f} | "
            f"Median rounds={median_rounds}"
        )

        return mean_ap

    return objective


# =============================================================================
# 5H — OPTIMIZE EACH HORIZON
# =============================================================================

optimization_results = {}

for horizon in OPTIMIZATION_HORIZONS:

    print("\n" + "=" * 105)
    print(f"OPTUNA — H{horizon}")
    print("=" * 105)

    db_path = OPT_ROOT / f"optuna_H{horizon}.db"

    study = optuna.create_study(
        study_name=f"XGB_V2_FLAT140_H{horizon}",
        storage=f"sqlite:///{db_path}",
        load_if_exists=True,
        direction="maximize",
        sampler=optuna.samplers.TPESampler(
            seed=MASTER_SEED + horizon,
            n_startup_trials=OPTUNA_STARTUP_TRIALS
        )
    )

    study.set_user_attr(
        "protocol_fingerprint",
        fold_fingerprint
    )

    completed = sum(
        t.state == optuna.trial.TrialState.COMPLETE
        for t in study.trials
    )

    remaining = max(
        0,
        N_TRIALS - len(study.trials)
    )

    print("Existing trials:", len(study.trials))
    print("Completed trials:", completed)
    print("Remaining trial slots:", remaining)

    if remaining > 0:

        study.optimize(
            make_objective(horizon),
            n_trials=remaining,
            n_jobs=1,
            gc_after_trial=True,
            show_progress_bar=True
        )

    completed_trials = [
        t for t in study.trials
        if t.state == optuna.trial.TrialState.COMPLETE
    ]

    if not completed_trials:
        raise RuntimeError(
            f"H{horizon}: no successfully completed trials."
        )

    best = study.best_trial

    results = {
        "horizon": horizon,
        "best_trial_number": best.number,
        "best_mean_cv_ap": float(best.value),
        "best_params": best.params,
        "fold_ap_std": best.user_attrs["fold_ap_std"],
        "fold_results": best.user_attrs["fold_results"],
        "fold_best_rounds": best.user_attrs["fold_best_rounds"],
        "selected_boost_rounds": best.user_attrs[
            "median_best_rounds"
        ],
        "n_trials_total": len(study.trials),
        "n_trials_complete": len(completed_trials)
    }

    optimization_results[f"H{horizon}"] = results

    output_path = (
        OPT_ROOT
        / f"BEST_TRAIN_CV_H{horizon}.json"
    )

    output_path.write_text(
        json.dumps(results, indent=2)
    )

    study.trials_dataframe().to_csv(
        OPT_ROOT / f"OPTUNA_TRIALS_H{horizon}.csv",
        index=False
    )

    print("\nBEST HORIZON CONFIGURATION")
    print(json.dumps(results, indent=2))

# =============================================================================
# 5I — SAVE FINAL OPTIMIZATION SUMMARY
# =============================================================================

summary = {
    "dataset": "Synthetic_NAF_v2",
    "version": "2.0.0",
    "representation": "FLAT140_LOG1P",
    "scope": "TRAIN_ONLY_GROUP_DISJOINT_CV",
    "generated_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "optimizer": "Optuna",
    "trials_per_horizon": N_TRIALS,
    "folds": N_SPLITS,
    "optimization_metric": "Mean sklearn Average Precision",
    "early_stopping_metric": "XGBoost AUCPR",
    "device": DEVICE,
    "results": optimization_results,
    "official_validation_used": False,
    "official_test_used": False,
    "final_models_trained": False
}

(
    OPT_ROOT / "TRAIN_CV_OPTIMIZATION_SUMMARY.json"
).write_text(
    json.dumps(summary, indent=2)
)

print("\n" + "=" * 105)
print("XGBOOST V2 — TRAIN-ONLY OPTIMIZATION COMPLETE")
print("=" * 105)

for horizon in OPTIMIZATION_HORIZONS:

    result = optimization_results[f"H{horizon}"]

    print(
        f"H{horizon}: "
        f"Best CV AP={result['best_mean_cv_ap']:.6f} | "
        f"CV SD={result['fold_ap_std']:.6f} | "
        f"Selected rounds={result['selected_boost_rounds']}"
    )

print("\nOptimization artifacts:", OPT_ROOT)
print("Full-TRAIN final models: NOT TRAINED")
print("Official VALIDATION: NOT USED")
print("Official TEST: NOT USED")
print("Frozen dataset: UNCHANGED")


In [ ]:

# ==========================================================================================
# CELL 6 — XGBOOST V2 FINAL FULL-TRAIN MODEL FITTING
#
# THIS CELL TRAINS TWO FINAL XGBOOST MODELS WHEN EXECUTED.
#
# Models:
#   XGBoost_V2_FLAT140_H5
#   XGBoost_V2_FLAT140_H10
#
# Uses:
#   Frozen Synthetic_NAF_v2 version 2.0.0
#   Full TRAIN partition only
#   Optuna-selected hyperparameters
#   Fixed CV-selected boosting rounds
#
# DOES NOT:
#   - Perform additional Optuna optimization
#   - Use official VALIDATION for early stopping
#   - Load or evaluate official TEST
#   - Change the frozen dataset
#   - Train H15, LSTM, or TCN models
# ==========================================================================================

from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import numpy as np
import xgboost as xgb

# ==========================================================================================
# 6A — PATHS
# ==========================================================================================

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/network-attack-forecasting"
)

DATA_DIR = (
    PROJECT_ROOT
    / "Synthetic_NAF_v2"
    / "release_candidate_corrected"
)

OPT_ROOT = (
    PROJECT_ROOT
    / "Synthetic_NAF_v2"
    / "models"
    / "XGBoost_V2_FLAT140"
    / "train_cv_optimization"
)

FINAL_MODEL_DIR = (
    PROJECT_ROOT
    / "Synthetic_NAF_v2"
    / "models"
    / "XGBoost_V2_FLAT140"
    / "final_train_models"
)

# Do not overwrite finalized model artifacts.
if FINAL_MODEL_DIR.exists() and any(FINAL_MODEL_DIR.iterdir()):
    raise RuntimeError(
        f"Final model folder is not empty: {FINAL_MODEL_DIR}. "
        "This cell will not overwrite an existing model release."
    )

# ==========================================================================================
# 6B — VERIFY TRAIN INPUTS
# ==========================================================================================

required_variables = [
    "X_FLAT",
    "Y_HORIZONS",
    "TEMPORAL_FEATURE_NAMES",
    "HORIZON_TO_COLUMN",
    "EPISODE_ID",
    "CV_GROUPS"
]

missing = [
    name for name in required_variables
    if name not in globals()
]

if missing:
    raise RuntimeError(
        "Run Cells 1–4 before final training. "
        f"Missing variables: {missing}"
    )

assert X_FLAT.shape == (88186, 140)
assert Y_HORIZONS.shape == (88186, 3)

assert np.isfinite(X_FLAT).all()

assert len(TEMPORAL_FEATURE_NAMES) == 140
assert len(np.unique(EPISODE_ID)) == 1400
assert len(np.unique(CV_GROUPS)) == 35

assert HORIZON_TO_COLUMN == {
    5: 0,
    10: 1,
    15: 2
}

# ==========================================================================================
# 6C — LOAD COMPLETED OPTUNA RESULTS
# ==========================================================================================

selected = {}

EXPECTED_SELECTED = {
    5: {
        "best_trial_number": 23,
        "selected_boost_rounds": 435
    },
    10: {
        "best_trial_number": 24,
        "selected_boost_rounds": 376
    }
}

for horizon in [5, 10]:

    path = OPT_ROOT / f"BEST_TRAIN_CV_H{horizon}.json"

    if not path.is_file():
        raise FileNotFoundError(
            f"Optuna selection artifact missing: {path}"
        )

    with open(path, "r") as f:
        result = json.load(f)

    if result["horizon"] != horizon:
        raise RuntimeError(
            f"H{horizon}: horizon identity mismatch."
        )

    if (
        result["best_trial_number"]
        != EXPECTED_SELECTED[horizon]["best_trial_number"]
    ):
        raise RuntimeError(
            f"H{horizon}: selected trial differs from completed study."
        )

    if (
        result["selected_boost_rounds"]
        != EXPECTED_SELECTED[horizon]["selected_boost_rounds"]
    ):
        raise RuntimeError(
            f"H{horizon}: boosting-round selection mismatch."
        )

    if result["n_trials_total"] != 25:
        raise RuntimeError(
            f"H{horizon}: unexpected optimization trial count."
        )

    if result["n_trials_complete"] < 24:
        raise RuntimeError(
            f"H{horizon}: insufficient successful optimization trials."
        )

    selected[horizon] = result

print("=" * 100)
print("XGBOOST V2 — FINAL FULL-TRAIN FITTING")
print("=" * 100)

for horizon in [5, 10]:

    result = selected[horizon]

    print(
        f"H{horizon}: "
        f"Best CV AP={result['best_mean_cv_ap']:.6f}, "
        f"Selected rounds={result['selected_boost_rounds']}"
    )

print("Official VALIDATION: NOT USED")
print("Official TEST: NOT USED")

# ==========================================================================================
# 6D — VERIFY FROZEN DATASET HASHES
# ==========================================================================================

HASH_FILE = DATA_DIR / "SHA256SUMS.json"

with open(HASH_FILE, "r") as f:
    frozen_hashes = json.load(f)

def sha256_file(path):

    digest = hashlib.sha256()

    with open(path, "rb") as stream:

        for block in iter(
            lambda: stream.read(1024 * 1024),
            b""
        ):
            digest.update(block)

    return digest.hexdigest()

for filename, expected_hash in frozen_hashes.items():

    path = DATA_DIR / filename

    if sha256_file(path) != expected_hash:
        raise RuntimeError(
            f"Frozen dataset modified: {filename}"
        )

print("Frozen dataset integrity: VERIFIED")

# ==========================================================================================
# 6E — RECONSTRUCT AND VERIFY FLAT140 FEATURES
# ==========================================================================================

with np.load(
    DATA_DIR / "windows_W10_TRAIN.npz",
    allow_pickle=False
) as pack:

    original_x = pack["X"].astype(np.float32)

    original_y = pack["y_horizons"].astype(np.int8)

expected_flat = np.log1p(
    original_x
).reshape(len(original_x), 140)

if not np.allclose(
    X_FLAT,
    expected_flat,
    rtol=1e-6,
    atol=1e-6
):
    raise RuntimeError(
        "FLAT140 feature alignment mismatch."
    )

if not np.array_equal(
    Y_HORIZONS,
    original_y
):
    raise RuntimeError(
        "TRAIN target alignment mismatch."
    )

del original_x, original_y, expected_flat

print("FLAT140 and forecasting labels: VERIFIED")

# ==========================================================================================
# 6F — FREEZE FINAL XGBOOST PARAMETERS
# ==========================================================================================

BASE_PARAMS = {
    "objective": "binary:logistic",
    "eval_metric": "aucpr",
    "tree_method": "hist",
    "device": "cpu",
    "max_bin": 256,
    "nthread": 2,
    "seed": 20261022,
    "verbosity": 0
}

MODELS = {}
MODEL_RECORDS = {}

# Build final directory only after all validation succeeds.
FINAL_MODEL_DIR.mkdir(parents=True, exist_ok=False)

# ==========================================================================================
# 6G — TRAIN FINAL H5 AND H10 MODELS
# ==========================================================================================

for horizon in [5, 10]:

    print("\n" + "=" * 100)
    print(f"FINAL TRAINING — H{horizon}")
    print("=" * 100)

    result = selected[horizon]

    target_col = HORIZON_TO_COLUMN[horizon]

    target = Y_HORIZONS[:, target_col].astype(np.int8)

    assert len(np.unique(target)) == 2

    params = {
        **BASE_PARAMS,
        **result["best_params"]
    }

    n_rounds = int(
        result["selected_boost_rounds"]
    )

    dtrain = xgb.DMatrix(
        X_FLAT,
        label=target,
        feature_names=TEMPORAL_FEATURE_NAMES
    )

    # No early stopping and no validation set.
    model = xgb.train(
        params=params,
        dtrain=dtrain,
        num_boost_round=n_rounds,
        evals=[],
        verbose_eval=False
    )

    if model.num_boosted_rounds() != n_rounds:
        raise RuntimeError(
            f"H{horizon}: unexpected trained boosting-round count."
        )

    model_path = (
        FINAL_MODEL_DIR
        / f"XGBoost_V2_FLAT140_H{horizon}.json"
    )

    model.save_model(
        str(model_path)
    )

    if not model_path.exists():
        raise RuntimeError(
            f"H{horizon}: model artifact was not saved."
        )

    record = {
        "model_name": f"XGBoost_V2_FLAT140_H{horizon}",
        "dataset": "Synthetic_NAF_v2",
        "dataset_version": "2.0.0",
        "training_split": "TRAIN",
        "training_rows": int(len(X_FLAT)),
        "unique_train_episodes": int(
            len(np.unique(EPISODE_ID))
        ),
        "input_shape": [10, 14],
        "representation": "FLAT140_LOG1P",
        "feature_names": TEMPORAL_FEATURE_NAMES,
        "forecast_horizon_minutes": horizon,
        "target_positive_prevalence": float(
            target.mean()
        ),
        "training_parameters": params,
        "boosting_rounds": n_rounds,
        "source_optuna_trial": result["best_trial_number"],
        "source_cv_average_precision": result["best_mean_cv_ap"],
        "source_cv_ap_std": result["fold_ap_std"],
        "model_sha256": sha256_file(model_path),
        "trained_utc": datetime.now(
            timezone.utc
        ).isoformat(),
        "official_validation_used": False,
        "official_test_used": False,
        "dataset_modified": False
    }

    record_path = (
        FINAL_MODEL_DIR
        / f"XGBoost_V2_FLAT140_H{horizon}_TRAIN_RECORD.json"
    )

    record_path.write_text(
        json.dumps(record, indent=2)
    )

    MODELS[horizon] = model
    MODEL_RECORDS[horizon] = record

    print(
        f"✓ H{horizon} model trained and saved"
    )

    print("Boosting rounds:", n_rounds)

    print(
        "TRAIN positive prevalence:",
        round(float(target.mean()), 6)
    )

    print(
        "Model SHA256:",
        record["model_sha256"]
    )

# ==========================================================================================
# 6H — SAVE FINAL MODEL REGISTRY
# ==========================================================================================

registry = {
    "dataset": "Synthetic_NAF_v2",
    "dataset_version": "2.0.0",
    "model_family": "XGBoost",
    "representation": "FLAT140_LOG1P",
    "training_status": "FINAL_FULL_TRAIN_COMPLETED",
    "models": {
        f"H{h}": {
            "model_file": f"XGBoost_V2_FLAT140_H{h}.json",
            "model_sha256": MODEL_RECORDS[h]["model_sha256"],
            "boosting_rounds": MODEL_RECORDS[h]["boosting_rounds"],
            "source_cv_average_precision": MODEL_RECORDS[h][
                "source_cv_average_precision"
            ],
            "official_validation_used": False,
            "official_test_used": False
        }
        for h in [5, 10]
    },
    "validation_status": "NOT_STARTED",
    "test_status": "EMBARGOED"
}

registry_path = (
    FINAL_MODEL_DIR
    / "XGBOOST_V2_FINAL_TRAIN_REGISTRY.json"
)

registry_path.write_text(
    json.dumps(registry, indent=2)
)

# ==========================================================================================
# 6I — VERIFY SAVED MODEL RELOAD
# ==========================================================================================

for horizon in [5, 10]:

    model_path = (
        FINAL_MODEL_DIR
        / f"XGBoost_V2_FLAT140_H{horizon}.json"
    )

    reloaded = xgb.Booster()
    reloaded.load_model(str(model_path))

    assert (
        reloaded.num_boosted_rounds()
        ==
        selected[horizon]["selected_boost_rounds"]
    )

    assert sha256_file(model_path) == MODEL_RECORDS[
        horizon
    ]["model_sha256"]

    print(
        f"✓ H{horizon} saved-model reload verified"
    )

# ==========================================================================================
# 6J — FINAL OUTPUT
# ==========================================================================================

print("\n" + "=" * 100)
print("XGBOOST V2 — FINAL FULL-TRAIN FITTING COMPLETE")
print("=" * 100)

for horizon in [5, 10]:

    record = MODEL_RECORDS[horizon]

    print(
        f"H{horizon}: "
        f"TRAIN rows={record['training_rows']:,} | "
        f"Rounds={record['boosting_rounds']} | "
        f"Prior CV AP={record['source_cv_average_precision']:.6f}"
    )

print("\nFinal model directory:")
print(FINAL_MODEL_DIR)

print("\nFinal XGBoost models: TRAINED AND SAVED")
print("Official VALIDATION: NOT USED")
print("Official TEST: NOT USED")
print("Frozen dataset: UNCHANGED")

print("\nNEXT: Review model artifacts before official validation.")


In [ ]:

# =============================================================================
# CELL 7 — XGBOOST V2 OFFICIAL VALIDATION
#
# EVALUATION ONLY — NO TRAINING
#
# Loads:
#   Frozen VALIDATION windows
#   Final XGBoost H5 and H10 models
#
# Reports:
#   Average Precision (AP)
#   ROC-AUC
#   Brier score
#   Calibration / ECE
#   Precision and recall at FIXED thresholds
#   False alerts on benign episodes
#   Source-specific performance
#   Episode-bootstrap confidence intervals
#   Comparison against prior TRAIN-only CV
#
# Never loads TEST.
# Never updates models or hyperparameters.
# =============================================================================

from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import numpy as np
import pandas as pd
import xgboost as xgb

from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    brier_score_loss,
    precision_score,
    recall_score,
    confusion_matrix
)

# =============================================================================
# 7A — PATHS
# =============================================================================

ROOT = Path(
    "/content/drive/MyDrive/network-attack-forecasting"
)

DATA_DIR = (
    ROOT / "Synthetic_NAF_v2" / "release_candidate_corrected"
)

FREEZE_DIR = (
    ROOT / "Synthetic_NAF_v2" / "FINAL_RELEASE_FREEZE"
)

MODEL_DIR = (
    ROOT
    / "Synthetic_NAF_v2"
    / "models"
    / "XGBoost_V2_FLAT140"
    / "final_train_models"
)

OPT_DIR = (
    ROOT
    / "Synthetic_NAF_v2"
    / "models"
    / "XGBoost_V2_FLAT140"
    / "train_cv_optimization"
)

REPORT_DIR = (
    ROOT
    / "Synthetic_NAF_v2"
    / "models"
    / "XGBoost_V2_FLAT140"
    / "official_validation"
)

VAL_FILE = DATA_DIR / "windows_W10_VALIDATION.npz"

META_FILE = (
    DATA_DIR
    / "episode_metadata_DO_NOT_USE_AS_PREDICTORS.csv"
)

# Fixed in advance; these are diagnostic thresholds,
# not validation-optimized operational thresholds.
FIXED_THRESHOLDS = [0.10, 0.20, 0.50]

# Episode-level bootstrap, not window-level bootstrap.
BOOTSTRAP_REPLICATES = 300
BOOTSTRAP_SEED = 20261022

HORIZONS_TO_EVALUATE = [5, 10]
HORIZON_TO_COLUMN = {5: 0, 10: 1, 15: 2}

# =============================================================================
# 7B — CHECK PRECONDITIONS
# =============================================================================

def read_json(path):
    with open(path, "r") as f:
        return json.load(f)


def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(
            lambda: f.read(1024 * 1024),
            b""
        ):
            h.update(block)
    return h.hexdigest()


def write_json(path, value):
    with open(path, "w") as f:
        json.dump(
            value,
            f,
            indent=2,
            default=str
        )


required_paths = [
    VAL_FILE,
    META_FILE,
    DATA_DIR / "FEATURE_AND_DATASET_CONTRACT.json",
    DATA_DIR / "SHA256SUMS.json",
    FREEZE_DIR / "Synthetic_NAF_v2_RELEASE_MANIFEST.json",
    FREEZE_DIR / "Synthetic_NAF_v2_FREEZE_LOCK.json",
    MODEL_DIR / "XGBOOST_V2_FINAL_TRAIN_REGISTRY.json"
]

for horizon in HORIZONS_TO_EVALUATE:
    required_paths.extend([
        MODEL_DIR / f"XGBoost_V2_FLAT140_H{horizon}.json",
        MODEL_DIR / f"XGBoost_V2_FLAT140_H{horizon}_TRAIN_RECORD.json",
        OPT_DIR / f"BEST_TRAIN_CV_H{horizon}.json"
    ])

missing = [str(p) for p in required_paths if not p.is_file()]

if missing:
    raise FileNotFoundError(
        "Missing required artifacts:\n" + "\n".join(missing)
    )

# One official assessment per model release.
# Reruns are not silently treated as new independent evaluations.
if REPORT_DIR.exists() and any(REPORT_DIR.iterdir()):
    raise RuntimeError(
        "Official validation report already exists:\n"
        f"{REPORT_DIR}\n"
        "Do not overwrite the existing official evaluation."
    )

CONTRACT = read_json(
    DATA_DIR / "FEATURE_AND_DATASET_CONTRACT.json"
)

RELEASE = read_json(
    FREEZE_DIR / "Synthetic_NAF_v2_RELEASE_MANIFEST.json"
)

LOCK = read_json(
    FREEZE_DIR / "Synthetic_NAF_v2_FREEZE_LOCK.json"
)

REGISTRY = read_json(
    MODEL_DIR / "XGBOOST_V2_FINAL_TRAIN_REGISTRY.json"
)

FEATURES = CONTRACT["features"]

assert len(FEATURES) == 14
assert CONTRACT["history_minutes"] == 10
assert CONTRACT["forecast_horizons"] == [5, 10, 15]

assert RELEASE["release_version"] == "2.0.0"
assert RELEASE["release_status"] == "FROZEN_CONTROLLED_SYNTHETIC_BENCHMARK"

canonical_manifest = json.dumps(
    RELEASE,
    sort_keys=True,
    separators=(",", ":"),
    ensure_ascii=False
).encode("utf-8")

assert hashlib.sha256(
    canonical_manifest
).hexdigest() == LOCK["manifest_sha256_canonical"]

assert REGISTRY["training_status"] == "FINAL_FULL_TRAIN_COMPLETED"
assert REGISTRY["validation_status"] == "NOT_STARTED"
assert REGISTRY["test_status"] == "EMBARGOED"

# Verify frozen dataset artifacts.
dataset_hashes = read_json(
    DATA_DIR / "SHA256SUMS.json"
)

for filename, expected_hash in dataset_hashes.items():
    if sha256_file(DATA_DIR / filename) != expected_hash:
        raise RuntimeError(
            f"Frozen dataset integrity failure: {filename}"
        )

print("=" * 100)
print("XGBOOST V2 — OFFICIAL VALIDATION")
print("=" * 100)
print("Dataset release: 2.0.0")
print("Frozen dataset hashes: VERIFIED")
print("Training: DISABLED")
print("Official TEST: NOT ACCESSED")

# =============================================================================
# 7C — LOAD VALIDATION DATA
# =============================================================================

with np.load(VAL_FILE, allow_pickle=False) as pack:

    X_VAL_SEQ = pack["X"].astype(np.float32)
    Y_VAL = pack["y_horizons"].astype(np.int8)

    VAL_IDS = (
        pd.Series(pack["episode_id"])
        .astype(str)
        .to_numpy()
    )

    VAL_MINUTES = pack["minute"].astype(int)
    VAL_ONSET = pack["onset_class"].astype(int)
    VAL_FAMILY = pack["family_class"].astype(int)

assert X_VAL_SEQ.shape[1:] == (10, 14)
assert Y_VAL.shape == (len(X_VAL_SEQ), 3)
assert len(VAL_IDS) == len(X_VAL_SEQ)
assert np.isfinite(X_VAL_SEQ).all()
assert (X_VAL_SEQ >= 0).all()

for horizon, col in HORIZON_TO_COLUMN.items():
    expected = (
        (VAL_ONSET >= 1) &
        (VAL_ONSET <= horizon)
    ).astype(np.int8)

    if not np.array_equal(Y_VAL[:, col], expected):
        raise RuntimeError(
            f"Validation H{horizon} label consistency failure"
        )

# Same exact preprocessing used during final training.
X_VAL_FLAT = np.log1p(
    X_VAL_SEQ
).reshape(len(X_VAL_SEQ), 140)

X_VAL_FLAT = np.ascontiguousarray(
    X_VAL_FLAT,
    dtype=np.float32
)

assert np.isfinite(X_VAL_FLAT).all()

FEATURE_NAMES = [
    f"t_minus_{9 - step}__{feature}"
    for step in range(10)
    for feature in FEATURES
]

assert len(FEATURE_NAMES) == 140

# =============================================================================
# 7D — MATCH EPISODE METADATA (NEVER AS MODEL INPUT)
# =============================================================================

meta = pd.read_csv(META_FILE)
meta["Episode_ID"] = meta["Episode_ID"].astype(str)

val_meta = (
    meta.loc[meta["Split"] == "VALIDATION"]
    .set_index("Episode_ID")
)

assert val_meta.index.is_unique
assert len(val_meta) == 300
assert set(VAL_IDS) == set(val_meta.index)

VAL_SOURCE = (
    val_meta.loc[VAL_IDS, "Source"]
    .astype(str)
    .to_numpy()
)

VAL_EPISODE_FAMILY = (
    val_meta.loc[VAL_IDS, "Family"]
    .astype(str)
    .to_numpy()
)

VAL_HARD_BENIGN = (
    val_meta.loc[VAL_IDS, "Hard_Benign"]
    .astype(bool)
    .to_numpy()
)

VAL_SPLITS = set(
    meta.loc[
        meta.Episode_ID.isin(VAL_IDS),
        "Split"
    ]
)

assert VAL_SPLITS == {"VALIDATION"}

print("\nVALIDATION INPUT")
print("Windows:", X_VAL_FLAT.shape)
print("Unique episodes:", len(np.unique(VAL_IDS)))
print("Sources:", np.unique(VAL_SOURCE).tolist())

for horizon in HORIZONS_TO_EVALUATE:
    col = HORIZON_TO_COLUMN[horizon]
    print(
        f"H{horizon} prevalence:",
        round(float(Y_VAL[:, col].mean()), 6)
    )

# =============================================================================
# 7E — METRIC FUNCTIONS
# =============================================================================

def safe_auc(y, score):
    if len(np.unique(y)) < 2:
        return np.nan
    return float(roc_auc_score(y, score))


def safe_ap(y, score):
    if len(np.unique(y)) < 2:
        return np.nan
    return float(average_precision_score(y, score))


def expected_calibration_error(y, score, bins=10):
    edges = np.linspace(0, 1, bins + 1)
    bin_ids = np.searchsorted(
        edges[1:-1],
        score,
        side="right"
    )

    total = len(y)
    ece = 0.0

    calibration_rows = []

    for bin_number in range(bins):
        selected = bin_ids == bin_number

        count = int(selected.sum())

        if count == 0:
            continue

        observed = float(np.mean(y[selected]))
        predicted = float(np.mean(score[selected]))

        ece += (
            count / total
        ) * abs(observed - predicted)

        calibration_rows.append({
            "bin": bin_number,
            "count": count,
            "mean_predicted_probability": predicted,
            "observed_positive_fraction": observed
        })

    return float(ece), calibration_rows


def threshold_metrics(y, score, threshold):

    pred = (score >= threshold).astype(np.int8)

    tn, fp, fn, tp = confusion_matrix(
        y,
        pred,
        labels=[0, 1]
    ).ravel()

    return {
        "threshold": float(threshold),
        "TP": int(tp),
        "FP": int(fp),
        "TN": int(tn),
        "FN": int(fn),
        "precision": float(
            precision_score(
                y,
                pred,
                zero_division=0
            )
        ),
        "recall": float(
            recall_score(
                y,
                pred,
                zero_division=0
            )
        ),
        "false_positive_rate": float(
            fp / (fp + tn)
        ) if (fp + tn) else np.nan
    }


def group_metrics(y, score, group, label):

    rows = []

    for value in sorted(np.unique(group)):

        mask = group == value

        rows.append({
            "group_type": label,
            "group": str(value),
            "windows": int(mask.sum()),
            "episodes": int(len(set(VAL_IDS[mask]))),
            "prevalence": float(y[mask].mean()),
            "AP": safe_ap(y[mask], score[mask]),
            "ROC_AUC": safe_auc(y[mask], score[mask]),
            "Brier": float(
                brier_score_loss(
                    y[mask],
                    score[mask]
                )
            )
        })

    return rows


# =============================================================================
# 7F — EPISODE-BOOTSTRAP CONFIDENCE INTERVALS
#
# Resample entire episodes, not individual W10 windows.
# These intervals describe variation among synthetic episodes;
# they do not establish real-world uncertainty.
# =============================================================================

def episode_bootstrap(
    y,
    score,
    episode_ids,
    n_replicates=300,
    seed=20261022
):

    rng = np.random.default_rng(seed)

    unique_ids, inverse = np.unique(
        episode_ids,
        return_inverse=True
    )

    episode_indices = [
        np.flatnonzero(inverse == i)
        for i in range(len(unique_ids))
    ]

    ap_values = []
    auc_values = []
    brier_values = []

    for _ in range(n_replicates):

        sampled = rng.integers(
            0,
            len(unique_ids),
            size=len(unique_ids)
        )

        sampled_indices = np.concatenate([
            episode_indices[i]
            for i in sampled
        ])

        yy = y[sampled_indices]
        ss = score[sampled_indices]

        if len(np.unique(yy)) < 2:
            continue

        ap_values.append(
            average_precision_score(yy, ss)
        )

        auc_values.append(
            roc_auc_score(yy, ss)
        )

        brier_values.append(
            brier_score_loss(yy, ss)
        )

    if not ap_values:
        raise RuntimeError(
            "No valid episode-bootstrap replicates."
        )

    def ci(values):
        return [
            float(np.quantile(values, 0.025)),
            float(np.quantile(values, 0.975))
        ]

    return {
        "method": "episode_cluster_bootstrap_percentile",
        "requested_replicates": n_replicates,
        "valid_replicates": len(ap_values),
        "AP_95_CI": ci(ap_values),
        "ROC_AUC_95_CI": ci(auc_values),
        "Brier_95_CI": ci(brier_values)
    }


# =============================================================================
# 7G — LOAD SAVED MODELS AND VERIFY HASHES
# =============================================================================

MODELS = {}
MODEL_RECORDS = {}

for horizon in HORIZONS_TO_EVALUATE:

    model_path = (
        MODEL_DIR /
        f"XGBoost_V2_FLAT140_H{horizon}.json"
    )

    record_path = (
        MODEL_DIR /
        f"XGBoost_V2_FLAT140_H{horizon}_TRAIN_RECORD.json"
    )

    record = read_json(record_path)

    model_hash = sha256_file(model_path)

    if model_hash != record["model_sha256"]:
        raise RuntimeError(
            f"H{horizon}: model hash mismatch"
        )

    if model_hash != REGISTRY["models"][f"H{horizon}"]["model_sha256"]:
        raise RuntimeError(
            f"H{horizon}: registry hash mismatch"
        )

    if record["forecast_horizon_minutes"] != horizon:
        raise RuntimeError(
            f"H{horizon}: model target mismatch"
        )

    if record["feature_names"] != FEATURE_NAMES:
        raise RuntimeError(
            f"H{horizon}: model feature order mismatch"
        )

    if record["training_split"] != "TRAIN":
        raise RuntimeError(
            f"H{horizon}: unexpected training split"
        )

    if record["official_validation_used"] is not False:
        raise RuntimeError(
            f"H{horizon}: validation was used in training"
        )

    model = xgb.Booster()
    model.load_model(str(model_path))

    if model.num_boosted_rounds() != record["boosting_rounds"]:
        raise RuntimeError(
            f"H{horizon}: saved tree count mismatch"
        )

    MODELS[horizon] = model
    MODEL_RECORDS[horizon] = record

    print(
        f"✓ H{horizon}: trained model integrity verified"
    )

# =============================================================================
# 7H — GENERATE VALIDATION PREDICTIONS (NO FITTING)
# =============================================================================

dval = xgb.DMatrix(
    X_VAL_FLAT,
    feature_names=FEATURE_NAMES
)

PREDICTIONS = {}

for horizon in HORIZONS_TO_EVALUATE:

    predictions = MODELS[horizon].predict(dval)

    assert predictions.shape == (len(X_VAL_FLAT),)
    assert np.isfinite(predictions).all()

    if not (
        (predictions >= 0).all()
        and
        (predictions <= 1).all()
    ):
        raise RuntimeError(
            f"H{horizon}: invalid probability predictions"
        )

    PREDICTIONS[horizon] = predictions

print("✓ Validation predictions generated")
print("✓ No models fitted or modified")

# =============================================================================
# 7I — OFFICIAL METRICS
# =============================================================================

overall_rows = []
source_rows = []
threshold_rows = []
calibration_rows = []
false_alert_rows = []
bootstrap_results = {}

summary_by_horizon = {}

for horizon in HORIZONS_TO_EVALUATE:

    col = HORIZON_TO_COLUMN[horizon]

    y = Y_VAL[:, col]
    p = PREDICTIONS[horizon]

    prevalence = float(y.mean())

    ap = safe_ap(y, p)
    auc = safe_auc(y, p)
    brier = float(brier_score_loss(y, p))

    ece, cal_rows = expected_calibration_error(
        y,
        p,
        bins=10
    )

    cv_ap = float(
        MODEL_RECORDS[horizon][
            "source_cv_average_precision"
        ]
    )

    overall_rows.append({
        "Horizon": f"H{horizon}",
        "Windows": int(len(y)),
        "Episodes": int(len(np.unique(VAL_IDS))),
        "Prevalence": prevalence,
        "AP": ap,
        "ROC_AUC": auc,
        "Brier": brier,
        "ECE_10_bins": ece,
        "Prior_Train_CV_AP": cv_ap,
        "Validation_AP_Minus_CV_AP": ap - cv_ap
    })

    summary_by_horizon[f"H{horizon}"] = {
        "AP": ap,
        "ROC_AUC": auc,
        "Brier": brier,
        "ECE": ece,
        "prevalence": prevalence,
        "previous_train_cv_ap": cv_ap
    }

    for row in cal_rows:
        row["horizon"] = f"H{horizon}"
        calibration_rows.append(row)

    source_metrics = group_metrics(
        y,
        p,
        VAL_SOURCE,
        "Source"
    )

    for row in source_metrics:
        row["Horizon"] = f"H{horizon}"
        source_rows.append(row)

    for threshold in FIXED_THRESHOLDS:

        stats = threshold_metrics(
            y,
            p,
            threshold
        )

        stats["Horizon"] = f"H{horizon}"
        threshold_rows.append(stats)

        # Benign-only false-alert evaluation.
        benign_mask = VAL_EPISODE_FAMILY == "BENIGN"

        for benign_type, type_mask in [
            (
                "HARD_BENIGN",
                benign_mask & VAL_HARD_BENIGN
            ),
            (
                "ORDINARY_BENIGN",
                benign_mask & ~VAL_HARD_BENIGN
            ),
            (
                "ALL_BENIGN",
                benign_mask
            )
        ]:

            ids_subset = VAL_IDS[type_mask]
            scores_subset = p[type_mask]

            alarms = scores_subset >= threshold

            unique_benign_ids = np.unique(ids_subset)

            alerted_episodes = sum(
                bool(
                    np.any(
                        alarms[ids_subset == episode_id]
                    )
                )
                for episode_id in unique_benign_ids
            )

            n_windows = int(type_mask.sum())
            n_alarms = int(alarms.sum())

            # These are window-level alert rates,
            # not independent alert events.
            exposure_hours = n_windows / 60.0

            false_alert_rows.append({
                "Horizon": f"H{horizon}",
                "Threshold": threshold,
                "Benign_Type": benign_type,
                "Episodes": len(unique_benign_ids),
                "Windows": n_windows,
                "False_Alert_Windows": n_alarms,
                "False_Alert_Window_Rate": (
                    n_alarms / n_windows
                    if n_windows else np.nan
                ),
                "Episodes_With_Any_Alert": alerted_episodes,
                "Fraction_Episodes_Alerted": (
                    alerted_episodes / len(unique_benign_ids)
                    if len(unique_benign_ids) else np.nan
                ),
                "Alerted_Window_Count_Per_Benign_Hour": (
                    n_alarms / exposure_hours
                    if exposure_hours else np.nan
                )
            })

    bootstrap_results[f"H{horizon}"] = episode_bootstrap(
        y=y,
        score=p,
        episode_ids=VAL_IDS,
        n_replicates=BOOTSTRAP_REPLICATES,
        seed=BOOTSTRAP_SEED + horizon
    )

# =============================================================================
# 7J — ADDITIONAL CONSISTENCY CHECKS
# =============================================================================

# Labels for H5 are nested inside labels for H10.
assert np.all(
    Y_VAL[:, 0] <= Y_VAL[:, 1]
)

# Two independently trained binary models need not produce
# nested probabilities. Measure this rather than silently
# modifying model outputs.
nesting_violation = (
    PREDICTIONS[5] > PREDICTIONS[10]
)

nested_probability_report = {
    "description": (
        "Fraction of validation windows where predicted "
        "P(H5) exceeds predicted P(H10). This violates "
        "the monotonicity expected of true horizon probabilities."
    ),
    "violation_count": int(nesting_violation.sum()),
    "violation_fraction": float(nesting_violation.mean()),
    "maximum_violation_magnitude": float(
        np.maximum(
            PREDICTIONS[5] - PREDICTIONS[10],
            0
        ).max()
    ),
    "probabilities_modified": False
}

# =============================================================================
# 7K — SAVE OFFICIAL VALIDATION ARTIFACTS
# =============================================================================

overall_df = pd.DataFrame(overall_rows)
source_df = pd.DataFrame(source_rows)
threshold_df = pd.DataFrame(threshold_rows)
calibration_df = pd.DataFrame(calibration_rows)
false_alert_df = pd.DataFrame(false_alert_rows)

prediction_df = pd.DataFrame({
    "Episode_ID": VAL_IDS,
    "Minute": VAL_MINUTES,
    "Source": VAL_SOURCE,
    "Episode_Family_METADATA_ONLY": VAL_EPISODE_FAMILY,
    "H5_Actual": Y_VAL[:, 0],
    "H10_Actual": Y_VAL[:, 1],
    "H5_Probability": PREDICTIONS[5],
    "H10_Probability": PREDICTIONS[10]
})

report = {
    "dataset": "Synthetic_NAF_v2",
    "version": "2.0.0",
    "model": "XGBoost_V2_FLAT140",
    "evaluation_split": "VALIDATION",
    "evaluation_status": "COMPLETED",
    "model_weights_modified": False,
    "model_training_performed": False,
    "hyperparameter_optimization_performed": False,
    "test_loaded": False,
    "validation_tuning_performed": False,
    "thresholds": FIXED_THRESHOLDS,
    "n_validation_windows": int(len(X_VAL_FLAT)),
    "n_validation_episodes": int(len(np.unique(VAL_IDS))),
    "model_hashes": {
        f"H{h}": MODEL_RECORDS[h]["model_sha256"]
        for h in HORIZONS_TO_EVALUATE
    },
    "overall_metrics": summary_by_horizon,
    "episode_bootstrap_CI": bootstrap_results,
    "horizon_probability_nesting": nested_probability_report,
    "methodological_notes": [
        "TRAIN group-disjoint CV was used for model selection.",
        "VALIDATION was not used for training or early stopping.",
        "Fixed thresholds are diagnostic and were not optimized.",
        "Bootstrap resampling is by episode, not individual window.",
        "False-alert window counts are not independent operational alert events.",
        "Results measure performance on a controlled synthetic benchmark.",
        "Official TEST remains embargoed."
    ],
    "generated_utc": datetime.now(
        timezone.utc
    ).isoformat()
}

# Create output directory only after successful computation.
REPORT_DIR.mkdir(
    parents=True,
    exist_ok=False
)

overall_df.to_csv(
    REPORT_DIR / "OFFICIAL_VALIDATION_METRICS.csv",
    index=False
)

source_df.to_csv(
    REPORT_DIR / "SOURCE_SPECIFIC_METRICS.csv",
    index=False
)

threshold_df.to_csv(
    REPORT_DIR / "FIXED_THRESHOLD_METRICS.csv",
    index=False
)

calibration_df.to_csv(
    REPORT_DIR / "CALIBRATION_10_BINS.csv",
    index=False
)

false_alert_df.to_csv(
    REPORT_DIR / "BENIGN_FALSE_ALERT_STATISTICS.csv",
    index=False
)

prediction_df.to_parquet(
    REPORT_DIR / "VALIDATION_PREDICTIONS.parquet",
    index=False
)

write_json(
    REPORT_DIR / "EPISODE_BOOTSTRAP_CONFIDENCE_INTERVALS.json",
    bootstrap_results
)

write_json(
    REPORT_DIR / "OFFICIAL_VALIDATION_REPORT.json",
    report
)

artifact_hashes = {
    p.name: sha256_file(p)
    for p in REPORT_DIR.iterdir()
    if p.is_file()
}

write_json(
    REPORT_DIR / "VALIDATION_ARTIFACT_SHA256SUMS.json",
    artifact_hashes
)

# =============================================================================
# 7L — FINAL CONSOLE REPORT
# =============================================================================

print("\n" + "=" * 100)
print("XGBOOST V2 — OFFICIAL VALIDATION COMPLETE")
print("=" * 100)

print("\nOVERALL METRICS")
print(
    overall_df.round(6).to_string(index=False)
)

print("\nSOURCE-SPECIFIC METRICS")
print(
    source_df[
        [
            "Horizon",
            "group",
            "windows",
            "prevalence",
            "AP",
            "ROC_AUC",
            "Brier"
        ]
    ].round(6).to_string(index=False)
)

print("\nFIXED-THRESHOLD METRICS")
print(
    threshold_df[
        [
            "Horizon",
            "threshold",
            "precision",
            "recall",
            "FP",
            "FN"
        ]
    ].round(6).to_string(index=False)
)

print("\nBENIGN FALSE ALERTS")
print(
    false_alert_df[
        [
            "Horizon",
            "Threshold",
            "Benign_Type",
            "Episodes_With_Any_Alert",
            "False_Alert_Windows",
            "Fraction_Episodes_Alerted"
        ]
    ].round(6).to_string(index=False)
)

print("\nEPISODE-BOOTSTRAP 95% CONFIDENCE INTERVALS")

for horizon in HORIZONS_TO_EVALUATE:
    ci = bootstrap_results[f"H{horizon}"]

    print(
        f"H{horizon} AP:",
        np.round(ci["AP_95_CI"], 6),
        "| ROC-AUC:",
        np.round(ci["ROC_AUC_95_CI"], 6)
    )

print("\nHORIZON PROBABILITY CONSISTENCY")

print(
    "P(H5) > P(H10) fraction:",
    round(
        nested_probability_report["violation_fraction"],
        6
    )
)

print("\nReports saved to:")
print(REPORT_DIR)

print("\nFinal model weights: UNCHANGED")
print("Additional training: NONE")
print("Hyperparameter optimization: NONE")
print("Official TEST: NOT ACCESSED")
print("Frozen dataset: UNCHANGED")

print("\n" + "=" * 100)
print("OFFICIAL VALIDATION COMPLETED SUCCESSFULLY")
print("=" * 100)
